# Stage 03b — Per-animal Optuna thresholds

Loads exactly one Stage 03a full WildFusion matrix. It builds the AnimalCLEF-style 70/70 open-set validation split from the full labelled database matrix, then runs 2,000 TPE trials over known-identity attachment, optional re-ranking, and Chinese Whispers.

The official Stage 00 five-seed `open_role` test queries are not used here. The output is a frozen `best_configs_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.json` for notebook 03c.

In [ ]:
%pip install -q optuna networkx scikit-learn

from collections import defaultdict
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import adjusted_rand_score, v_measure_score

# ============================================================
# USER KNOBS — must match the 03a artifact
# ============================================================
DATASET_NAME = "jaguar_antonio"
GLOBAL_DESCRIPTOR = "ApexID_BHCA_A_v2"
N_TRIALS = 2000
OPTUNA_SEED = 42
SPLIT_SEED = 42
KNOWN_IDENTITY_FRAC = 0.7
DB_FRAC_WITHIN_KNOWN = 0.7
MIN_IMAGES_PER_KNOWN_IDENTITY = 3
# Match the AnimalCLEF objective: reranking is always enabled and its parameters are tuned.
USE_RERANK = True

ALLOWED_DATASETS = ("bristol_gorillas", "spac_wild", "elephants", "gcn_id", "jaguar_antonio")
ALLOWED_GLOBAL_DESCRIPTORS = (
    "ApexID_BHCA_A_v2", "ApexID_BHCA_SCA_v2", "ApexID_BHCA_SCSF2_v2", "MiewIDv3", "MegaDescriptor",
)
assert DATASET_NAME in ALLOWED_DATASETS
assert GLOBAL_DESCRIPTOR in ALLOWED_GLOBAL_DESCRIPTORS

artifact_name = f"sim_matrix_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}_full.npy"
candidates = [
    Path(f"/kaggle/working/apexid_cvpr_stage03/03a/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}/{artifact_name}"),
    Path(os.environ.get("APEXID_STAGE03A_MATRIX", "")),
]
if Path("/kaggle/input").is_dir():
    candidates.extend(Path("/kaggle/input").glob(f"**/{artifact_name}"))
MATRIX_PATH = next((p.resolve() for p in candidates if str(p) and p.is_file()), None)
if MATRIX_PATH is None:
    raise FileNotFoundError(f"Attach the 03a output containing {artifact_name}")
OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE03B_OUTPUT",
    f"/kaggle/working/apexid_cvpr_stage03/03b/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}",
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MATRIX_PATH:", MATRIX_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# Load and validate the full train/train artifact.
obj = np.load(MATRIX_PATH, allow_pickle=True).item()
if DATASET_NAME not in obj:
    raise KeyError(f"{MATRIX_PATH} has keys {list(obj)}; expected {DATASET_NAME}")
bundle = obj[DATASET_NAME]
sim_full = np.asarray(bundle["sim"], dtype=np.float32)
image_ids = np.asarray(bundle["image_ids"]).astype(str)
identities = np.asarray(bundle["identities"]).astype(str)
if bundle.get("global_descriptor") != GLOBAL_DESCRIPTOR:
    raise ValueError(f"Artifact model mismatch: {bundle.get('global_descriptor')} != {GLOBAL_DESCRIPTOR}")
if sim_full.shape != (len(image_ids), len(image_ids)) or len(identities) != len(image_ids):
    raise ValueError("03a matrix/metadata shapes are inconsistent")
if not np.isfinite(sim_full).all() or sim_full.min() < 0 or sim_full.max() > 1:
    raise ValueError("03a similarities must be finite and clipped to [0,1]")
if len(set(image_ids)) != len(image_ids):
    raise ValueError("03a image_ids are not unique")
print("Full matrix:", sim_full.shape, sim_full.dtype, "identities:", len(set(identities)))


def build_open_set_split(similarity, labels, ids, seed):
    rng = np.random.default_rng(seed)
    identity_to_positions = {
        identity: np.flatnonzero(labels == identity).astype(np.int64)
        for identity in np.unique(labels)
    }
    eligible = np.array(
        [identity for identity, pos in identity_to_positions.items() if len(pos) >= MIN_IMAGES_PER_KNOWN_IDENTITY],
        dtype=object,
    )
    if len(eligible) < 2:
        raise ValueError("Need at least two eligible identities for a leak-safe open-set split")
    n_known = min(len(eligible) - 1, max(1, int(round(len(eligible) * KNOWN_IDENTITY_FRAC))))
    known = set(rng.choice(eligible, size=n_known, replace=False).tolist())
    db, query_known, query_unknown = [], [], []
    for identity, positions in identity_to_positions.items():
        positions = positions.copy()
        rng.shuffle(positions)
        if identity in known:
            n_db = min(len(positions) - 1, max(1, int(round(len(positions) * DB_FRAC_WITHIN_KNOWN))))
            db.extend(positions[:n_db])
            query_known.extend(positions[n_db:])
        else:
            query_unknown.extend(positions)
    db = np.asarray(sorted(db), dtype=np.int64)
    q_known = np.asarray(sorted(query_known), dtype=np.int64)
    q_unknown = np.asarray(sorted(query_unknown), dtype=np.int64)
    query = np.concatenate([q_known, q_unknown])
    return {
        "sim_query_query": similarity[np.ix_(query, query)],
        "sim_query_db": similarity[np.ix_(query, db)],
        "y_true_query": labels[query],
        "y_true_db": labels[db],
        "query_image_ids": ids[query],
        "db_image_ids": ids[db],
        "query_positions": query,
        "db_positions": db,
        "known_identities": np.asarray(sorted(known), dtype=str),
        "unknown_identities": np.asarray(sorted(set(identity_to_positions) - known), dtype=str),
        "known_mask_query": np.isin(labels[query], list(known)),
        "unknown_mask_query": ~np.isin(labels[query], list(known)),
    }

split = build_open_set_split(sim_full, identities, image_ids, SPLIT_SEED)
print({k: v.shape for k, v in split.items() if isinstance(v, np.ndarray)})
db_positions = set(split["db_positions"].tolist())
query_positions = set(split["query_positions"].tolist())
if db_positions & query_positions or db_positions | query_positions != set(range(len(image_ids))):
    raise RuntimeError("Optuna database/query positions must be disjoint and cover the full matrix")
if not set(split["query_image_ids"]).isdisjoint(set(split["db_image_ids"])):
    raise RuntimeError("Optuna query/database image leakage")
if set(split["y_true_query"][split["unknown_mask_query"]]) & set(split["y_true_db"]):
    raise RuntimeError("Unknown-query identities leaked into the Optuna database")
for identity in split["known_identities"]:
    if identity not in set(split["y_true_db"]) or identity not in set(split["y_true_query"]):
        raise RuntimeError(f"Known identity {identity!r} lacks a database or query sample")
if split["sim_query_query"].shape != (len(query_positions), len(query_positions)):
    raise RuntimeError("Query-query matrix shape mismatch")
if split["sim_query_db"].shape != (len(query_positions), len(db_positions)):
    raise RuntimeError("Query-database matrix shape mismatch")
np.savez_compressed(
    OUTPUT_ROOT / f"optuna_split_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.npz",
    **{k: v for k, v in split.items() if isinstance(v, np.ndarray)},
)

In [ ]:
# AnimalCLEF identity scoring + graph clustering (ported from MIA-AI-Team/AnimalCLEF26).
def compute_identity_scores(sim_query_db, db_ids, top_r):
    identity_names = list(dict.fromkeys(db_ids.tolist()))
    scores = np.zeros((sim_query_db.shape[0], len(identity_names)), dtype=np.float32)
    for j, identity in enumerate(identity_names):
        cols = np.flatnonzero(db_ids == identity)
        sims = sim_query_db[:, cols]
        r = min(int(top_r), sims.shape[1])
        scores[:, j] = sims.mean(axis=1) if r == sims.shape[1] else np.partition(sims, -r, axis=1)[:, -r:].mean(axis=1)
    return np.asarray(identity_names, dtype=object), scores


def summarize_best_identities(names, scores):
    best_idx = scores.argmax(axis=1)
    best_scores = scores[np.arange(len(scores)), best_idx]
    best_names = names[best_idx]
    if scores.shape[1] >= 2:
        top2 = np.partition(scores, -2, axis=1)[:, -2:]
        second = np.min(top2, axis=1)
    else:
        second = np.zeros(len(scores), dtype=np.float32)
    return best_names, best_scores, second, best_scores - second


def get_topk_neighbors(sim_matrix, top_k):
    n = sim_matrix.shape[0]
    k = min(int(top_k), n - 1)
    if k <= 0:
        return np.empty((n, 0), dtype=np.int64)
    sims = sim_matrix.copy()
    np.fill_diagonal(sims, -np.inf)
    return np.argpartition(sims, -k, axis=1)[:, -k:]


def chinese_whispers(sim_matrix, threshold, iterations, top_k, k_core_degree, seed=42, skip_mutual_knn=False):
    random.seed(seed)
    np.random.seed(seed)
    n = sim_matrix.shape[0]
    adj = sim_matrix.copy()
    np.fill_diagonal(adj, 0)
    adj[adj < threshold] = 0
    graph = nx.from_numpy_array(adj)
    if graph.number_of_edges():
        try:
            backbone = nx.k_core(graph, k=int(k_core_degree))
            if backbone.number_of_nodes() == 0:
                backbone = graph
        except Exception:
            backbone = graph
    else:
        backbone = graph
    backbone_nodes = list(backbone.nodes()) or list(range(n))
    in_backbone = np.zeros(n, dtype=bool)
    in_backbone[backbone_nodes] = True
    valid = (sim_matrix >= threshold) & in_backbone[:, None] & in_backbone[None, :]
    np.fill_diagonal(valid, False)
    if not skip_mutual_knn:
        topk = get_topk_neighbors(sim_matrix, top_k)
        nearest = np.zeros((n, n), dtype=bool)
        if topk.shape[1]:
            nearest[np.arange(n)[:, None], topk] = True
        valid &= nearest & nearest.T
    sources, targets = np.nonzero(valid)
    graph_list = [[] for _ in range(n)]
    for source, target in zip(sources, targets):
        graph_list[int(source)].append((int(target), float(sim_matrix[source, target])))
    labels = list(range(n))
    nodes = backbone_nodes.copy()
    for _ in range(int(iterations)):
        random.shuffle(nodes)
        changes = 0
        for node in nodes:
            weights = defaultdict(float)
            for neighbor, weight in graph_list[node]:
                weights[labels[neighbor]] += weight
            if weights:
                new_label = max(weights.items(), key=lambda item: (item[1], -item[0]))[0]
                changes += int(new_label != labels[node])
                labels[node] = new_label
        if changes == 0:
            break
    return np.unique(labels, return_inverse=True)[1].astype(np.int64)


def final_open_set_labels(cluster_ids, best_names, best_scores, margins, tau_attach, tau_margin):
    votes = defaultdict(lambda: defaultdict(float))
    for i, cluster in enumerate(cluster_ids):
        if best_scores[i] >= tau_attach and margins[i] >= tau_margin:
            votes[int(cluster)][str(best_names[i])] += float(best_scores[i])
    cluster_to_known = {
        cluster: max(identity_votes.items(), key=lambda item: item[1])[0]
        for cluster, identity_votes in votes.items() if identity_votes
    }
    return np.asarray([
        cluster_to_known.get(int(cluster), f"__new_cluster_{int(cluster)}")
        for cluster in cluster_ids
    ], dtype=str)

In [ ]:
# Exact k-reciprocal implementation used by the AnimalCLEF repository.
def re_ranking(sim_matrix, k1=20, k2=6, lambda_value=0.3):
    sim_matrix = np.asarray(sim_matrix, dtype=np.float32)
    n = sim_matrix.shape[0]
    original_dist = np.clip(1.0 - sim_matrix, 0.0, None)
    col_max = np.max(original_dist, axis=0)
    col_max[col_max == 0] = 1.0
    original_dist = (original_dist / col_max).T.astype(np.float32)
    values = np.zeros((n, n), dtype=np.float32)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)
    half_k1 = int(np.around(k1 / 2.0)) + 1
    for i in range(n):
        forward = initial_rank[i, : k1 + 1]
        backward = initial_rank[forward, : k1 + 1]
        reciprocal = forward[np.where(backward == i)[0]]
        expanded = reciprocal.copy()
        for q in reciprocal:
            q_forward = initial_rank[q, :half_k1]
            q_backward = initial_rank[q_forward, :half_k1]
            q_recip = q_forward[np.where(q_backward == q)[0]]
            if len(q_recip) and len(np.intersect1d(q_recip, reciprocal)) > (2.0 / 3.0) * len(q_recip):
                expanded = np.append(expanded, q_recip)
        expanded = np.unique(expanded)
        weight = np.exp(-original_dist[i, expanded])
        if weight.sum() > 0:
            values[i, expanded] = weight / weight.sum()
    if k2 > 1:
        expanded_values = np.zeros_like(values)
        for i in range(n):
            expanded_values[i] = np.mean(values[initial_rank[i, :k2]], axis=0)
        values = expanded_values
    jaccard = np.zeros((n, n), dtype=np.float32)
    inverse = [np.where(values[:, j] != 0)[0] for j in range(n)]
    for i in range(n):
        temp_min = np.zeros(n, dtype=np.float32)
        nonzero = np.where(values[i] != 0)[0]
        for column in nonzero:
            rows = inverse[column]
            temp_min[rows] += np.minimum(values[i, column], values[rows, column])
        jaccard[i] = 1.0 - temp_min / (2.0 - temp_min + 1e-12)
    final_dist = np.clip(jaccard * (1.0 - lambda_value) + original_dist * lambda_value, 0.0, 1.0)
    np.fill_diagonal(final_dist, 0.0)
    return (1.0 - final_dist).astype(np.float32)

In [ ]:
sim_qq = split["sim_query_query"]
sim_qg = split["sim_query_db"]
y_query = split["y_true_query"]
y_db = split["y_true_db"]


def objective(trial):
    top_r = trial.suggest_int("top_r", 1, 5)
    tau_attach = trial.suggest_float("tau_attach", 0.10, 0.90, step=0.01)
    tau_margin = trial.suggest_float("tau_margin", 0.10, 0.90, step=0.01)
    cw_threshold = trial.suggest_float("cw_threshold", 0.12, 0.68, step=0.01)
    cw_top_k = trial.suggest_int("cw_top_k", 7, 60)
    cw_iterations = trial.suggest_int("cw_iterations", 5, 30)
    cw_k_core_degree = trial.suggest_int("cw_k_core_degree", 1, 5)
    rr_k1 = trial.suggest_int("rr_k1", 5, 50)
    rr_k2 = trial.suggest_int("rr_k2", 1, 15)
    rr_lambda = trial.suggest_float("rr_lambda", 0.10, 0.90)
    use_rerank = USE_RERANK

    identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, top_r)
    best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
    sim_for_cw = re_ranking(sim_qq, rr_k1, rr_k2, rr_lambda) if use_rerank else sim_qq
    clusters = chinese_whispers(
        sim_for_cw,
        threshold=cw_threshold,
        iterations=cw_iterations,
        top_k=cw_top_k,
        k_core_degree=cw_k_core_degree,
        seed=42,
        skip_mutual_knn=use_rerank,
    )
    predicted = final_open_set_labels(clusters, best_names, best_scores, margins, tau_attach, tau_margin)
    ari = adjusted_rand_score(y_query, predicted)
    v_measure = v_measure_score(y_query, predicted)
    trial.set_user_attr("ari", float(ari))
    trial.set_user_attr("v_measure", float(v_measure))
    return 0.6 * ari + 0.4 * v_measure


storage_path = OUTPUT_ROOT / f"optuna_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.sqlite3"
study = optuna.create_study(
    study_name=f"{GLOBAL_DESCRIPTOR}__{DATASET_NAME}",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED, multivariate=True),
    storage=f"sqlite:///{storage_path.as_posix()}",
    load_if_exists=True,
)
completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
remaining = max(0, N_TRIALS - completed_trials)
print("Completed trials:", completed_trials, "remaining:", remaining)
if remaining:
    study.optimize(objective, n_trials=remaining, gc_after_trial=True, show_progress_bar=True)
print("Best value:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
# Freeze and independently reproduce the winning trial through the same graph.
best_params = dict(study.best_params)
best_params["use_rerank"] = USE_RERANK

identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, int(best_params["top_r"]))
best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
best_sim_for_cw = re_ranking(
    sim_qq,
    int(best_params["rr_k1"]),
    int(best_params["rr_k2"]),
    float(best_params["rr_lambda"]),
)
best_clusters = chinese_whispers(
    best_sim_for_cw,
    threshold=float(best_params["cw_threshold"]),
    iterations=int(best_params["cw_iterations"]),
    top_k=int(best_params["cw_top_k"]),
    k_core_degree=int(best_params["cw_k_core_degree"]),
    seed=42,
    skip_mutual_knn=True,
)
best_predicted = final_open_set_labels(
    best_clusters, best_names, best_scores, margins,
    float(best_params["tau_attach"]), float(best_params["tau_margin"]),
)
recomputed_ari = float(adjusted_rand_score(y_query, best_predicted))
recomputed_v_measure = float(v_measure_score(y_query, best_predicted))
recomputed_objective = 0.6 * recomputed_ari + 0.4 * recomputed_v_measure
if not np.isclose(recomputed_objective, float(study.best_value), rtol=0.0, atol=1e-12):
    raise RuntimeError(
        f"Winning trial is not reproducible: stored={study.best_value}, recomputed={recomputed_objective}"
    )

completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
best_config = {
    **best_params,
    "objective": "0.6*ARI+0.4*V-measure",
    "objective_value": recomputed_objective,
    "validation_ari": recomputed_ari,
    "validation_v_measure": recomputed_v_measure,
    "n_trials_completed": int(completed_trials),
    "split_seed": SPLIT_SEED,
    "known_identity_frac": KNOWN_IDENTITY_FRAC,
    "db_frac_within_known": DB_FRAC_WITHIN_KNOWN,
    "min_images_per_known_identity": MIN_IMAGES_PER_KNOWN_IDENTITY,
    "dataset": DATASET_NAME,
    "global_descriptor": GLOBAL_DESCRIPTOR,
}
config_path = OUTPUT_ROOT / f"best_configs_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.json"
config_path.write_text(json.dumps({DATASET_NAME: best_config}, indent=2, sort_keys=True), encoding="utf-8")
study.trials_dataframe().to_csv(OUTPUT_ROOT / "optuna_trials.csv", index=False)

split_path = OUTPUT_ROOT / f"optuna_split_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.npz"
manifest = {
    "stage": "03b_optuna",
    "status": "passed",
    "dataset": DATASET_NAME,
    "global_descriptor": GLOBAL_DESCRIPTOR,
    "source_matrix": str(MATRIX_PATH),
    "source_matrix_sha256": hashlib.sha256(MATRIX_PATH.read_bytes()).hexdigest(),
    "optuna_split_sha256": hashlib.sha256(split_path.read_bytes()).hexdigest(),
    "test_split_used_for_tuning": False,
    "winning_trial_recomputed": True,
    "rerank_mutual_knn_policy": "skip_mutual_knn=True after reranking",
    "config": best_config,
}
(OUTPUT_ROOT / "stage03b_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({DATASET_NAME: best_config}, indent=2))
print(f"\nNEXT: Save {OUTPUT_ROOT} as a Kaggle dataset and attach it with the 03a matrix to notebook 03c.")